In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import json
import pickle
import random
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

SEEDS = [42, 123, 2024]

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

SEQ_DIR = BASE_PROJECT / "processed_intra_sequence_features_plus_base"
OUT_DIR = BASE_PROJECT / "results_intra_sequence_cross_attention_plus_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["emodb", "ravdess", "resd"]

LABELS = ["angry", "disgust", "fear", "happy", "neutral", "sad"]

ID_TO_LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL_TO_ID = {label: i for i, label in ID_TO_LABEL.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", DEVICE)
print("SEQ_DIR:", SEQ_DIR)
print("OUT_DIR:", OUT_DIR)

DEVICE: cuda
SEQ_DIR: /content/drive/MyDrive/New Jurnal Cross/processed_intra_sequence_features_plus_base
OUT_DIR: /content/drive/MyDrive/New Jurnal Cross/results_intra_sequence_cross_attention_plus_base


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "uar": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def make_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=LABELS,
        labels=list(range(len(LABELS))),
        zero_division=0,
        output_dict=True
    )
    return pd.DataFrame(report).transpose()


def save_confusion_matrix_csv(cm, out_path):
    df_cm = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    df_cm.to_csv(out_path, index=True)


def compute_class_weights(y_train, n_classes=6):
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    weights = counts.sum() / (n_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

In [4]:
class SequenceFusionDataset(Dataset):
    def __init__(self, X_e2v, X_hc, mask, y):
        self.X_e2v = torch.tensor(X_e2v, dtype=torch.float32)
        self.X_hc = torch.tensor(X_hc, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_e2v[idx], self.X_hc[idx], self.mask[idx], self.y[idx]


def make_sequence_loader(X_e2v, X_hc, mask, y, batch_size=16, shuffle=False):
    dataset = SequenceFusionDataset(X_e2v, X_hc, mask, y)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False
    )


def load_sequence_dataset(dataset_name):
    ds_dir = SEQ_DIR / dataset_name

    data = {}

    for split in ["train", "val", "test"]:
        data[f"X_e2v_{split}"] = np.load(ds_dir / f"X_e2v_seq_{split}.npy").astype(np.float32)
        data[f"X_hc_{split}"] = np.load(ds_dir / f"X_hc_seq_{split}.npy").astype(np.float32)
        data[f"mask_{split}"] = np.load(ds_dir / f"mask_{split}.npy").astype(np.float32)
        data[f"y_{split}"] = np.load(ds_dir / f"y_{split}.npy").astype(np.int64)
        data[f"meta_{split}"] = pd.read_csv(ds_dir / f"meta_{split}.csv")

    data["e2v_dim"] = data["X_e2v_train"].shape[-1]
    data["hc_dim"] = data["X_hc_train"].shape[-1]
    data["target_frames"] = data["X_e2v_train"].shape[1]

    return data

In [5]:
for ds in DATASETS:
    data = load_sequence_dataset(ds)

    print("=" * 80)
    print(ds.upper())
    print("E2V train:", data["X_e2v_train"].shape)
    print("HC train :", data["X_hc_train"].shape)
    print("Mask     :", data["mask_train"].shape)
    print("y        :", data["y_train"].shape)
    print("E2V dim  :", data["e2v_dim"])
    print("HC dim   :", data["hc_dim"])

EMODB
E2V train: (498, 200, 768)
HC train : (498, 200, 43)
Mask     : (498, 200)
y        : (498,)
E2V dim  : 768
HC dim   : 43
RAVDESS
E2V train: (704, 200, 768)
HC train : (704, 200, 43)
Mask     : (704, 200)
y        : (704,)
E2V dim  : 768
HC dim   : 43
RESD
E2V train: (873, 200, 768)
HC train : (873, 200, 43)
Mask     : (873, 200)
y        : (873,)
E2V dim  : 768
HC dim   : 43


In [6]:
class AttentivePooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, x, mask=None):
        """
        x: [B, T, D]
        mask: [B, T], 1 valid, 0 padding
        """
        scores = self.attn(x).squeeze(-1)  # [B, T]

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = torch.softmax(scores, dim=1)  # [B, T]
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)

        return pooled, weights


def masked_mean_pool(x, mask):
    mask = mask.unsqueeze(-1)
    x = x * mask
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return x.sum(dim=1) / denom


def masked_max_pool(x, mask):
    mask = mask.unsqueeze(-1)
    x = x.masked_fill(mask == 0, -1e9)
    return x.max(dim=1).values

In [7]:
class SequenceCrossAttentionFusion(nn.Module):
    def __init__(
        self,
        e2v_dim=768,
        hc_dim=43,
        d_model=128,
        num_heads=4,
        num_classes=6,
        dropout=0.30
    ):
        super().__init__()

        self.e2v_proj = nn.Sequential(
            nn.Linear(e2v_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.hc_proj = nn.Sequential(
            nn.Linear(hc_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.e2v_to_hc = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.hc_to_e2v = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm_e = nn.LayerNorm(d_model)
        self.norm_h = nn.LayerNorm(d_model)

        self.pool_e = AttentivePooling(d_model)
        self.pool_h = AttentivePooling(d_model)

        # pooled features:
        # e_attentive, h_attentive, mean_e, mean_h, max_e, max_h, |e-h|, e*h
        fusion_dim = d_model * 8

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, num_classes)
        )

    def forward(self, x_e2v, x_hc, mask=None, return_attention=False):
        """
        x_e2v: [B, T, 768]
        x_hc : [B, T, 43]
        mask : [B, T], 1 valid, 0 pad
        """
        e = self.e2v_proj(x_e2v)  # [B, T, D]
        h = self.hc_proj(x_hc)    # [B, T, D]

        key_padding_mask = None
        if mask is not None:
            key_padding_mask = mask == 0  # True means ignore

        e_att, attn_eh = self.e2v_to_hc(
            query=e,
            key=h,
            value=h,
            key_padding_mask=key_padding_mask,
            need_weights=True
        )

        h_att, attn_he = self.hc_to_e2v(
            query=h,
            key=e,
            value=e,
            key_padding_mask=key_padding_mask,
            need_weights=True
        )

        e_fused = self.norm_e(e + e_att)
        h_fused = self.norm_h(h + h_att)

        e_att_pool, e_pool_weights = self.pool_e(e_fused, mask)
        h_att_pool, h_pool_weights = self.pool_h(h_fused, mask)

        e_mean = masked_mean_pool(e_fused, mask)
        h_mean = masked_mean_pool(h_fused, mask)

        e_max = masked_max_pool(e_fused, mask)
        h_max = masked_max_pool(h_fused, mask)

        diff = torch.abs(e_att_pool - h_att_pool)
        prod = e_att_pool * h_att_pool

        fusion = torch.cat(
            [
                e_att_pool,
                h_att_pool,
                e_mean,
                h_mean,
                e_max,
                h_max,
                diff,
                prod
            ],
            dim=1
        )

        logits = self.classifier(fusion)

        if return_attention:
            return logits, {
                "attn_eh": attn_eh,
                "attn_he": attn_he,
                "pool_e": e_pool_weights,
                "pool_h": h_pool_weights
            }

        return logits

In [8]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for x_e2v, x_hc, mask, y_batch in loader:
        x_e2v = x_e2v.to(DEVICE)
        x_hc = x_hc.to(DEVICE)
        mask = mask.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(x_e2v, x_hc, mask)
            loss = criterion(logits, y_batch)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        total_loss += loss.item() * x_e2v.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(y_batch.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(np.array(all_targets), np.array(all_preds))

    return avg_loss, metrics, np.array(all_targets), np.array(all_preds)


@torch.no_grad()
def predict_model(model, loader):
    model.eval()

    all_preds = []
    all_targets = []
    all_probs = []

    for x_e2v, x_hc, mask, y_batch in loader:
        x_e2v = x_e2v.to(DEVICE)
        x_hc = x_hc.to(DEVICE)
        mask = mask.to(DEVICE)

        logits = model(x_e2v, x_hc, mask)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(y_batch.numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    return np.array(all_targets), np.array(all_preds), np.array(all_probs)

In [9]:
def train_eval_sequence_xattn(
    dataset_name,
    seed,
    batch_size=16,
    lr=3e-4,
    weight_decay=1e-4,
    max_epochs=120,
    patience=15,
    d_model=128,
    num_heads=4,
    dropout=0.30
):
    set_seed(seed)

    data = load_sequence_dataset(dataset_name)

    train_loader = make_sequence_loader(
        data["X_e2v_train"],
        data["X_hc_train"],
        data["mask_train"],
        data["y_train"],
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = make_sequence_loader(
        data["X_e2v_val"],
        data["X_hc_val"],
        data["mask_val"],
        data["y_val"],
        batch_size=batch_size,
        shuffle=False
    )

    test_loader = make_sequence_loader(
        data["X_e2v_test"],
        data["X_hc_test"],
        data["mask_test"],
        data["y_test"],
        batch_size=batch_size,
        shuffle=False
    )

    model = SequenceCrossAttentionFusion(
        e2v_dim=data["e2v_dim"],
        hc_dim=data["hc_dim"],
        d_model=d_model,
        num_heads=num_heads,
        num_classes=len(LABELS),
        dropout=dropout
    ).to(DEVICE)

    class_weights = compute_class_weights(data["y_train"], n_classes=len(LABELS)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5
    )

    best_val_macro_f1 = -1.0
    best_epoch = -1
    best_state = None
    no_improve = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss, train_metrics, _, _ = run_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer
        )

        val_loss, val_metrics, _, _ = run_one_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None
        )

        scheduler.step(val_metrics["macro_f1"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "lr": optimizer.param_groups[0]["lr"]
        }
        history.append(row)

        current = val_metrics["macro_f1"]

        if current > best_val_macro_f1:
            best_val_macro_f1 = current
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"[{dataset_name} | seed={seed}] "
                f"Epoch {epoch:03d} | "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val_loss:.4f} | "
                f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
                f"best={best_val_macro_f1:.4f}"
            )

        if no_improve >= patience:
            print(
                f"[{dataset_name} | seed={seed}] Early stopping at epoch {epoch}. "
                f"Best epoch={best_epoch}, best val macro-F1={best_val_macro_f1:.4f}"
            )
            break

    model.load_state_dict(best_state)

    y_val_true, y_val_pred, y_val_prob = predict_model(model, val_loader)
    y_test_true, y_test_pred, y_test_prob = predict_model(model, test_loader)

    val_metrics = compute_metrics(y_val_true, y_val_pred)
    test_metrics = compute_metrics(y_test_true, y_test_pred)

    run_dir = OUT_DIR / dataset_name / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), run_dir / "sequence_cross_attention_model.pt")

    with open(run_dir / "training_config.json", "w") as f:
        json.dump({
            "dataset": dataset_name,
            "seed": seed,
            "batch_size": batch_size,
            "lr": lr,
            "weight_decay": weight_decay,
            "max_epochs": max_epochs,
            "patience": patience,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "e2v_dim": data["e2v_dim"],
            "hc_dim": data["hc_dim"],
            "target_frames": data["target_frames"],
            "d_model": d_model,
            "num_heads": num_heads,
            "dropout": dropout,
            "model": "SequenceCrossAttentionFusion",
            "fusion_type": "temporal_bidirectional_cross_attention"
        }, f, indent=2)

    pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)

    pd.DataFrame([{
        "dataset": dataset_name,
        "seed": seed,
        "split": "val",
        "best_epoch": best_epoch,
        **val_metrics
    }]).to_csv(run_dir / "val_metrics.csv", index=False)

    pd.DataFrame([{
        "dataset": dataset_name,
        "seed": seed,
        "split": "test",
        "best_epoch": best_epoch,
        **test_metrics
    }]).to_csv(run_dir / "test_metrics.csv", index=False)

    make_report_df(y_val_true, y_val_pred).to_csv(run_dir / "val_classification_report.csv")
    make_report_df(y_test_true, y_test_pred).to_csv(run_dir / "test_classification_report.csv")

    cm_val = confusion_matrix(y_val_true, y_val_pred, labels=list(range(len(LABELS))))
    cm_test = confusion_matrix(y_test_true, y_test_pred, labels=list(range(len(LABELS))))

    save_confusion_matrix_csv(cm_val, run_dir / "val_confusion_matrix.csv")
    save_confusion_matrix_csv(cm_test, run_dir / "test_confusion_matrix.csv")

    pred_val_df = data["meta_val"].copy()
    pred_val_df["y_true"] = y_val_true
    pred_val_df["y_pred"] = y_val_pred
    pred_val_df["true_label"] = [ID_TO_LABEL[i] for i in y_val_true]
    pred_val_df["pred_label"] = [ID_TO_LABEL[i] for i in y_val_pred]
    for i, label in enumerate(LABELS):
        pred_val_df[f"prob_{label}"] = y_val_prob[:, i]
    pred_val_df.to_csv(run_dir / "val_predictions.csv", index=False)

    pred_test_df = data["meta_test"].copy()
    pred_test_df["y_true"] = y_test_true
    pred_test_df["y_pred"] = y_test_pred
    pred_test_df["true_label"] = [ID_TO_LABEL[i] for i in y_test_true]
    pred_test_df["pred_label"] = [ID_TO_LABEL[i] for i in y_test_pred]
    for i, label in enumerate(LABELS):
        pred_test_df[f"prob_{label}"] = y_test_prob[:, i]
    pred_test_df.to_csv(run_dir / "test_predictions.csv", index=False)

    row_val = {
        "dataset": dataset_name,
        "seed": seed,
        "split": "val",
        "best_epoch": best_epoch,
        **val_metrics
    }

    row_test = {
        "dataset": dataset_name,
        "seed": seed,
        "split": "test",
        "best_epoch": best_epoch,
        **test_metrics
    }

    return row_val, row_test

In [10]:
all_rows = []

for dataset_name in DATASETS:
    print("=" * 100)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 100)

    for seed in SEEDS:
        print(f"\nTraining sequence cross-attention | dataset={dataset_name} | seed={seed}")

        row_val, row_test = train_eval_sequence_xattn(
            dataset_name=dataset_name,
            seed=seed,
            batch_size=16,
            lr=3e-4,
            weight_decay=1e-4,
            max_epochs=120,
            patience=15,
            d_model=128,
            num_heads=4,
            dropout=0.30
        )

        all_rows.append(row_val)
        all_rows.append(row_test)

        print("VAL :", {k: round(v, 4) for k, v in row_val.items() if isinstance(v, float)})
        print("TEST:", {k: round(v, 4) for k, v in row_test.items() if isinstance(v, float)})

results = pd.DataFrame(all_rows)
results.to_csv(OUT_DIR / "all_seed_results.csv", index=False)

display(results)
print("Saved:", OUT_DIR / "all_seed_results.csv")

DATASET: EMODB

Training sequence cross-attention | dataset=emodb | seed=42
[emodb | seed=42] Epoch 001 | train_loss=0.8007 | val_loss=0.9459 | val_macro_f1=0.7214 | best=0.7214
[emodb | seed=42] Epoch 010 | train_loss=0.2327 | val_loss=0.7838 | val_macro_f1=0.8005 | best=0.8005
[emodb | seed=42] Epoch 020 | train_loss=0.0761 | val_loss=1.0183 | val_macro_f1=0.8129 | best=0.8280
[emodb | seed=42] Epoch 030 | train_loss=0.0088 | val_loss=1.1185 | val_macro_f1=0.8387 | best=0.8387
[emodb | seed=42] Epoch 040 | train_loss=0.0053 | val_loss=1.1610 | val_macro_f1=0.7797 | best=0.8387
[emodb | seed=42] Early stopping at epoch 44. Best epoch=29, best val macro-F1=0.8387
VAL : {'accuracy': 0.8451, 'macro_f1': 0.8387, 'weighted_f1': 0.8371, 'uar': 0.8515}
TEST: {'accuracy': 0.906, 'macro_f1': 0.9052, 'weighted_f1': 0.9059, 'uar': 0.9027}

Training sequence cross-attention | dataset=emodb | seed=123
[emodb | seed=123] Epoch 001 | train_loss=0.7798 | val_loss=0.9429 | val_macro_f1=0.7205 | best=0

,dataset,seed,split,best_epoch,accuracy,macro_f1,weighted_f1,uar
0,emodb,42,val,29,0.845070,0.838657,0.837115,0.851496
1,emodb,42,test,29,0.906040,0.905243,0.905890,0.902701
2,emodb,123,val,17,0.816901,0.814242,0.809626,0.824786
3,emodb,123,test,17,0.912752,0.911304,0.912798,0.908737
4,emodb,2024,val,28,0.845070,0.844304,0.842500,0.847650
5,emodb,2024,test,28,0.912752,0.910278,0.912500,0.909230
6,ravdess,42,val,10,0.931818,0.924392,0.933836,0.937500
7,ravdess,42,test,10,0.982955,0.981877,0.982984,0.984375
8,ravdess,123,val,5,0.931818,0.925601,0.933192,0.937500
9,ravdess,123,test,5,0.982955,0.981633,0.982896,0.979167


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_sequence_cross_attention_plus_base/all_seed_results.csv


In [11]:
metrics = ["accuracy", "macro_f1", "weighted_f1", "uar"]

summary_rows = []

for dataset_name in DATASETS:
    for split in ["val", "test"]:
        sub = results[
            (results["dataset"] == dataset_name) &
            (results["split"] == split)
        ]

        row = {
            "dataset": dataset_name,
            "split": split,
            "n_seeds": len(sub),
            "best_epoch_mean": sub["best_epoch"].mean(),
            "best_epoch_std": sub["best_epoch"].std(ddof=1),
        }

        for metric in metrics:
            row[f"{metric}_mean"] = sub[metric].mean()
            row[f"{metric}_std"] = sub[metric].std(ddof=1)

        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "summary_mean_std.csv", index=False)

display(summary)
print("Saved:", OUT_DIR / "summary_mean_std.csv")

,dataset,split,n_seeds,best_epoch_mean,best_epoch_std,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,uar_mean,uar_std
0,emodb,val,3,24.666667,6.658328,0.835681,0.016263,0.832401,0.015978,0.829747,0.017632,0.841311,0.014439
1,emodb,test,3,24.666667,6.658328,0.910515,0.003875,0.908942,0.003244,0.910396,0.003905,0.906889,0.003635
2,ravdess,val,3,7.333333,2.516611,0.931818,0.000000,0.924988,0.000605,0.933177,0.000667,0.937500,0.000000
3,ravdess,test,3,7.333333,2.516611,0.977273,0.009841,0.975524,0.010793,0.977215,0.009916,0.973958,0.013780
4,resd,val,3,2.333333,0.577350,0.657706,0.008213,0.652275,0.009394,0.650383,0.009925,0.659625,0.004905
5,resd,test,3,2.333333,0.577350,0.599520,0.004154,0.577726,0.005755,0.601578,0.002962,0.600277,0.002707


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_sequence_cross_attention_plus_base/summary_mean_std.csv


In [12]:
SVM_DIR = BASE_PROJECT / "results_intra_handcrafted_svm"
HC_MLP_DIR = BASE_PROJECT / "results_intra_handcrafted_mlp"
E2V_MLP_DIR = BASE_PROJECT / "results_intra_emotion2vec_mlp_plus_base"
CONCAT_DIR = BASE_PROJECT / "results_intra_concat_fusion_mlp_plus_base"
MODAL_XATTN_DIR = BASE_PROJECT / "results_intra_cross_attention_fusion_plus_base"
SEQ_XATTN_DIR = BASE_PROJECT / "results_intra_sequence_cross_attention_plus_base"

svm_summary = pd.read_csv(SVM_DIR / "summary_mean_std.csv")
hc_mlp_summary = pd.read_csv(HC_MLP_DIR / "summary_mean_std.csv")
e2v_summary = pd.read_csv(E2V_MLP_DIR / "summary_mean_std.csv")
concat_summary = pd.read_csv(CONCAT_DIR / "summary_mean_std.csv")
modal_xattn_summary = pd.read_csv(MODAL_XATTN_DIR / "summary_mean_std.csv")
seq_xattn_summary = pd.read_csv(SEQ_XATTN_DIR / "summary_mean_std.csv")

def get_test(df, name):
    x = df[df["split"] == "test"].copy()
    x["model"] = name
    return x

compare = pd.concat(
    [
        get_test(svm_summary, "Handcrafted SVM-RBF"),
        get_test(hc_mlp_summary, "Handcrafted MLP"),
        get_test(e2v_summary, "emotion2vec MLP"),
        get_test(concat_summary, "Concat Fusion MLP"),
        get_test(modal_xattn_summary, "Modality-level Cross-Attention"),
        get_test(seq_xattn_summary, "Sequence-level Cross-Attention"),
    ],
    ignore_index=True
)

cols = [
    "model",
    "dataset",
    "accuracy_mean",
    "accuracy_std",
    "uar_mean",
    "uar_std",
    "macro_f1_mean",
    "macro_f1_std",
    "weighted_f1_mean",
    "weighted_f1_std",
]

compare = compare[cols]
compare.to_csv(SEQ_XATTN_DIR / "compare_all_intra_test.csv", index=False)

display(compare)
print("Saved:", SEQ_XATTN_DIR / "compare_all_intra_test.csv")

,model,dataset,accuracy_mean,accuracy_std,uar_mean,uar_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std
0,Handcrafted SVM-RBF,emodb,0.765101,0.000000e+00,0.771862,0.000000,0.770845,0.000000,0.766550,0.000000
1,Handcrafted SVM-RBF,ravdess,0.585227,0.000000e+00,0.572917,0.000000,0.562632,0.000000,0.575213,0.000000
2,Handcrafted SVM-RBF,resd,0.230216,0.000000e+00,0.216041,0.000000,0.177384,0.000000,0.201610,0.000000
3,Handcrafted MLP,emodb,0.722595,5.125924e-02,0.722497,0.057131,0.724663,0.053999,0.723408,0.052484
4,Handcrafted MLP,ravdess,0.564394,3.280399e-03,0.576389,0.003007,0.556696,0.007691,0.556044,0.004402
5,Handcrafted MLP,resd,0.213429,3.244065e-02,0.205268,0.026294,0.208880,0.041376,0.228542,0.040862
6,emotion2vec MLP,emodb,0.914989,1.397091e-02,0.914272,0.015576,0.915618,0.014075,0.914662,0.013859
7,emotion2vec MLP,ravdess,0.977273,1.359740e-16,0.968750,0.000000,0.973412,0.000000,0.977055,0.000000
8,emotion2vec MLP,resd,0.565947,1.497601e-02,0.567849,0.019836,0.546025,0.017867,0.566678,0.015246
9,Concat Fusion MLP,emodb,0.932886,1.162450e-02,0.932959,0.011840,0.934209,0.011421,0.933224,0.011423


Saved: /content/drive/MyDrive/New Jurnal Cross/results_intra_sequence_cross_attention_plus_base/compare_all_intra_test.csv
